In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    when,
    col,
    lit,
    current_date
)

inventory = spark.table(
    "agentdb.gold.fact_inventory_snapshot"
)

snapshot_inventory_health = (

    inventory

    .withColumn(
        "inventory_health_score",
        when(
            col("days_of_cover") >= 38,
            lit(100.0)
        )
        .when(
            col("days_of_cover") >= 37,
            lit(75.0)
        )
        .when(
            col("days_of_cover") >= 36,
            lit(50.0)
        )
        .otherwise(
            lit(25.0)
        )
    )

    .withColumn(
        "risk_level",
        when(
            col("days_of_cover") < 36,
            "CRITICAL"
        )
        .when(
            col("days_of_cover") < 37,
            "HIGH"
        )
        .when(
            col("days_of_cover") < 38,
            "MEDIUM"
        )
        .otherwise(
            "LOW"
        )
    )

    .withColumn(
        "snapshot_date",
        current_date()
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )

    .select(
        "snapshot_date",
        "product_key",
        "store_key",
        "inventory_qty",
        "days_of_cover",
        "inventory_health_score",
        "risk_level",
        "created_at"
    )
)

(
    snapshot_inventory_health
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.snapshot_inventory_health"
    )
)

display(
    snapshot_inventory_health
    .groupBy("risk_level")
    .count()
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    current_date,
    when,
    col,
    lit
)

suppliers = spark.table(
    "agentdb.gold.dim_supplier"
)

disruptions = spark.table(
    "agentdb.gold.fact_supplier_disruptions"
)

supplier_risk = (

    suppliers.alias("s")
    .drop("created_at")

    .join(
        disruptions.alias("d").drop("created_at"),
        "supplier_key",
        "left"
    )

    .withColumn(
        "disruption_probability",

        when(
            col("d.supplier_key").isNotNull(),
            lit(0.80)
        ).otherwise(
            lit(0.10)
        )
    )

    .withColumn(
        "lead_time_risk",

        when(
            col("lead_time_days") > 10,
            lit(0.80)
        )
        .when(
            col("lead_time_days") > 7,
            lit(0.50)
        )
        .otherwise(
            lit(0.20)
        )
    )

    .withColumn(
        "snapshot_date",
        current_date()
    )

    .withColumn(
        "created_at",
        current_timestamp()
    )

    .select(
        "snapshot_date",
        col("s.supplier_key").alias("supplier_key"),
        col("s.risk_score").alias("risk_score"),
        "disruption_probability",
        "lead_time_risk",
        "created_at"
    )
)

(
    supplier_risk
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.gold.snapshot_supplier_risk"
    )
)

display(
    supplier_risk
    .orderBy(
        col("risk_score").desc()
    )
)